# 02 — Prophet Model
**Niloo**

In [26]:
import pandas as pd
import numpy as np
from prophet import Prophet
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "vscode"

In [27]:
df_raw = pd.read_csv("/content/clean_data.csv")
df_raw['date'] = pd.to_datetime(df_raw['date'])

print("Shape:", df_raw.shape)
print("Columns:", df_raw.columns.tolist())
print("Range:", df_raw['date'].min(), "->", df_raw['date'].max())
print(df_raw.head())

Shape: (6820, 6)
Columns: ['date', 'temp_min', 'quality_min', 'temp_max', 'quality_max', 'quality_flag']
Range: 2007-06-01 00:00:00 -> 2026-01-31 00:00:00
        date  temp_min quality_min  temp_max quality_max  quality_flag
0 2016-02-02       4.1           Y       7.3           Y             1
1 2016-02-03       2.6           Y       6.0           Y             1
2 2016-02-04       2.6           Y       4.2           Y             1
3 2016-02-05       2.0           Y       3.5           Y             1
4 2016-02-06       2.7           Y       5.0           Y             1


In [28]:
df = pd.DataFrame({
    'ds': df_raw['date'],
    'y': (df_raw['temp_min'] + df_raw['temp_max']) / 2
})

df = df.sort_values('ds').reset_index(drop=True)

print("Shape:", df.shape)
print("NaN in y:", df['y'].isna().sum())
print("Temp range:", f"{df['y'].min():.1f}°C to {df['y'].max():.1f}°C")
print(df.head())

Shape: (6820, 2)
NaN in y: 0
Temp range: -10.2°C to 24.6°C
          ds      y
0 2007-06-01  13.60
1 2007-06-02  15.05
2 2007-06-03  15.20
3 2007-06-04  17.25
4 2007-06-05  18.20


In [29]:
assert 'ds' in df.columns and 'y' in df.columns
assert pd.api.types.is_datetime64_any_dtype(df['ds'])
assert pd.api.types.is_numeric_dtype(df['y'])
assert df['ds'].is_monotonic_increasing
assert df['y'].isna().sum() == 0
print("All checks passed ✓")

All checks passed ✓


In [30]:
model = Prophet(
    interval_width=0.95,
    daily_seasonality=False,  # vi har dagsmedel, inte tim-data
    yearly_seasonality=True,  # viktigt för temperatur
    weekly_seasonality=False  # temperatur har inget veckomönster
)
model.fit(df)

In [31]:
# 5 års daglig prediktion framåt (2026-02-01 → 2031-01-31)
future = model.make_future_dataframe(periods=5*365, freq='D')
forecast = model.predict(future)

print("Forecast shape:", forecast.shape)
print(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail())

Forecast shape: (8645, 16)
             ds      yhat  yhat_lower  yhat_upper
8640 2031-01-26  1.753634   -3.445391    7.002651
8641 2031-01-27  1.747647   -3.675259    7.143328
8642 2031-01-28  1.740061   -3.525144    6.783735
8643 2031-01-29  1.730615   -3.686445    6.768306
8644 2031-01-30  1.719202   -3.277251    6.832274


In [32]:
fig = go.Figure()

# Historik
fig.add_trace(go.Scatter(
    x=df['ds'], y=df['y'],
    name='Historisk data',
    mode='markers',
    marker=dict(size=2, color='#888')
))

# Prediktion
fig.add_trace(go.Scatter(
    x=forecast['ds'], y=forecast['yhat'],
    name='Prophet prediktion',
    line=dict(color='#00E5FF', width=2)
))

# Konfidensintervall
fig.add_trace(go.Scatter(
    x=forecast['ds'], y=forecast['yhat_upper'],
    name='Övre 95%',
    line=dict(color='rgba(0,229,255,0.2)', width=0),
    showlegend=False
))
fig.add_trace(go.Scatter(
    x=forecast['ds'], y=forecast['yhat_lower'],
    name='95% intervall',
    line=dict(color='rgba(0,229,255,0.2)', width=0),
    fill='tonexty',
    fillcolor='rgba(0,229,255,0.15)'
))

fig.update_layout(
    title='Temperaturprediktion 2026–2031 — Prophet',
    xaxis_title='Datum',
    yaxis_title='Temperatur (°C)',
    template='plotly_dark',
    hovermode='x unified'
)

fig

In [33]:
model = Prophet(interval_width=0.95, daily_seasonality=False)
model.fit(df)

In [34]:
future = model.make_future_dataframe(periods=1461, freq='D')
forecast = model.predict(future)

print(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail())

             ds      yhat  yhat_lower  yhat_upper
8276 2030-01-27  1.726149   -3.589927    6.800745
8277 2030-01-28  1.677847   -3.409691    6.609597
8278 2030-01-29  1.667543   -3.357096    6.624383
8279 2030-01-30  1.689729   -3.520713    7.117545
8280 2030-01-31  1.654296   -3.486271    6.864463


In [36]:
fig = go.Figure()
import plotly.io as pio

# Historisk data
fig.add_trace(go.Scatter(
    x=df['ds'], y=df['y'],
    name='Historisk data',
    line=dict(color='#90A4AE', width=1)
))

# Konfidensintervall
fig.add_trace(go.Scatter(
    x=pd.concat([forecast['ds'], forecast['ds'][::-1]]),
    y=pd.concat([forecast['yhat_upper'], forecast['yhat_lower'][::-1]]),
    fill='toself',
    fillcolor='rgba(0, 229, 255, 0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    name='Konfidensintervall 95%'
))

# Prediktion
fig.add_trace(go.Scatter(
    x=forecast['ds'], y=forecast['yhat'],
    name='Prophet prediktion',
    line=dict(color='#00E5FF', width=2)
))

fig.update_layout(
    title='Temperaturprediktion 2026–2030 — nordcast',
    xaxis_title='Datum',
    yaxis_title='Temperatur (°C)',
    template='plotly_dark'
)

pio.renderers.default = "notebook"   # or "vscode"

fig.show()

In [37]:
future_only = forecast[forecast['ds'] > '2026-01-31'].copy()
future_only['year'] = future_only['ds'].dt.year

summary = future_only.groupby('year')[['yhat', 'yhat_lower', 'yhat_upper']].mean().round(2)
print(summary)

       yhat  yhat_lower  yhat_upper
year                               
2026  10.64        5.68       15.64
2027   9.91        4.92       14.92
2028   9.90        4.90       14.90
2029   9.92        4.87       14.98
2030   2.02       -3.00        7.13


In [38]:
os.makedirs("../models", exist_ok=True)
joblib.dump(model, "../models/prophet_model.pkl")
print("Modell sparad!")

Modell sparad!


In [ ]:
print(df['ds'].dt.year.value_counts().sort_index())

ds
2007    214
2008    366
2009    365
2010    365
2011    365
2012    366
2013    365
2014    365
2015    365
2016    366
2017    365
2018    365
2019    365
2020    366
2021    365
2022    365
2023    365
2024    366
2025    365
2026     31
Name: count, dtype: int64


In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

historical = forecast[forecast['ds'] <= '2026-01-31'].copy()
historical = historical.merge(df, on='ds')

mae = mean_absolute_error(historical['y'], historical['yhat'])
rmse = np.sqrt(mean_squared_error(historical['y'], historical['yhat']))

print(f"MAE:  {mae:.2f}°C")
print(f"RMSE: {rmse:.2f}°C")

MAE:  2.00°C
RMSE: 2.57°C


In [13]:
from google.colab import files
files.download("../models/prophet_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
import joblib
import os

os.makedirs("/content/models", exist_ok=True)
joblib.dump(model, "/content/models/prophet_model.pkl")
print("Storlek:", os.path.getsize("/content/models/prophet_model.pkl"), "bytes")

Storlek: 670017 bytes
